# AV1 - Analise e Visualizacao de Dados (2026.2)
## Grupo 6 - EcoVolt | **Notebook 1 de 3 - V1: composicao da demanda critica**

**Objetivo.** Mostrar, equipamento por equipamento, como se forma a **potencia critica** (`Pcritica`) de cada cenario de continuidade energetica.

**Pergunta (V1).** *Quais cargas formam a demanda critica?*

**Estrutura do notebook.** 1) Coleta -> 2) Tratamento -> 3) Limpeza (5 verificacoes) -> 4) Indicador -> 5) Grafico -> 6) Por que este grafico -> 7) Leitura e limitacoes.


## 1. Coleta de dados

**Fonte:** `analise/cenarios.json`, bloco `cenarios` (cargas de cada cenario).

**Classificacao (exigencia do professor):** dado **simulado/declarado** pelo grupo. Nao existe base operacional real de cargas criticas; os valores seguem faixas publicas de equipamentos e sao **hipotese**, nao medicao.


In [1]:
import json
from pathlib import Path

import pandas as pd

def encontrar_raiz(inicio=None):
    """Sobe a arvore de diretorios ate achar a pasta do projeto (dados/ + analise/)."""
    p = Path(inicio or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "dados").exists() and (cand / "analise").exists():
            return cand
    raise FileNotFoundError("pasta do projeto nao encontrada")


BASE = encontrar_raiz()
RAW = BASE / "dados" / "raw"
PROC = BASE / "dados" / "processados"
VISTAS = BASE / "dashboard" / "vistas"
PROC.mkdir(parents=True, exist_ok=True)
VISTAS.mkdir(parents=True, exist_ok=True)
print("Raiz do projeto:", BASE)

cfg = json.loads((BASE / "analise" / "cenarios.json").read_text(encoding="utf-8"))
print("Titulo :", cfg["meta"]["titulo"])
print("Decisao:", cfg["meta"]["decisao"])
for cen in cfg["cenarios"]:
    print(f"  - {cen['id']:20s} -> {len(cen['cargas']):2d} equipamento(s), horizonte {cen['duracao_h']:.0f} h")


Raiz do projeto: C:\Users\pedro\OneDrive\Desktop\Análise
Titulo : Cenarios sinteticos declarados - Continuidade energetica com banco de 2a vida
Decisao: Nas condicoes informadas, o cenario atende as cargas criticas durante o tempo requerido de interrupcao?
  - posto_vacinas        ->  3 equipamento(s), horizonte 8 h
  - telecom_erb          ->  2 equipamento(s), horizonte 4 h
  - residencia_essencial ->  4 equipamento(s), horizonte 6 h
  - sem_carga_critica    ->  0 equipamento(s), horizonte 4 h


## 2. Tratamento

A fonte guarda as cargas como **lista aninhada**. Para analisar, "explodimos" em uma tabela **longa** (1 linha = 1 equipamento em 1 cenario) e derivamos o subtotal de potencia `subtotal_kW = potencia_kW * qtd`.


In [2]:
linhas = []
for cen in cfg["cenarios"]:
    for item in cen["cargas"]:
        linhas.append({
            "cenario_id": cen["id"],
            "cenario": cen["nome"],
            "duracao_h": cen["duracao_h"],
            "equipamento": item["equipamento"],
            "potencia_kW": item["potencia_kW"],
            "qtd": item["qtd"],
        })

cargas = pd.DataFrame(linhas)
cargas["potencia_kW"] = pd.to_numeric(cargas["potencia_kW"], errors="coerce")
cargas["qtd"] = pd.to_numeric(cargas["qtd"], errors="coerce").astype("Int64")
cargas["subtotal_kW"] = (cargas["potencia_kW"] * cargas["qtd"]).round(4)
cargas


,cenario_id,cenario,duracao_h,equipamento,potencia_kW,qtd,subtotal_kW
0,posto_vacinas,Sala de vacinas (posto de saude),8.0,Refrigerador de vacinas,0.20,1,0.2
1,posto_vacinas,Sala de vacinas (posto de saude),8.0,Iluminacao de emergencia,0.05,4,0.2
2,posto_vacinas,Sala de vacinas (posto de saude),8.0,Terminal + impressora,0.10,1,0.1
3,telecom_erb,Estacao radio-base (telecom),4.0,Transmissor/radio,0.40,1,0.4
4,telecom_erb,Estacao radio-base (telecom),4.0,Ar-condicionado de emergencia,0.80,1,0.8
5,residencia_essencial,Residencia - circuito essencial,6.0,Geladeira,0.15,1,0.15
6,residencia_essencial,Residencia - circuito essencial,6.0,Bomba d'agua,0.50,1,0.5
7,residencia_essencial,Residencia - circuito essencial,6.0,Iluminacao,0.08,3,0.24
8,residencia_essencial,Residencia - circuito essencial,6.0,Roteador/internet,0.02,1,0.02


## 3. Limpeza - as cinco verificacoes

Comecamos sempre pelas verificacoes; **so limpamos o que a verificacao aponta**. O original (`cenarios.json`) nunca e alterado.


In [3]:
def verificar(df, chave):
    """As cinco verificacoes de qualidade (bloco base)."""
    return {
        "1. chaves/duplicidade":
            f"{int(df.duplicated().sum())} linhas duplicadas; "
            f"{int(df.duplicated(subset=chave).sum())} duplicadas na chave {tuple(chave)}",
        "2. ausencia/codigos":
            f"{int(df.isna().sum().sum())} celulas nulas",
        "3. tipos/unidades":
            ", ".join(f"{c}={t}" for c, t in df.dtypes.astype(str).items()),
    }


def dominio_cargas(df):
    return {
        "4. dominio/plausibilidade": (
            f"potencia min = {df['potencia_kW'].min():.2f} kW (todas > 0: {bool((df['potencia_kW'] > 0).all())}); "
            f"qtd min = {int(df['qtd'].min())} (todas >= 1: {bool((df['qtd'] >= 1).all())}); "
            f"duracao min = {df['duracao_h'].min():.0f} h"),
        "5. cobertura/juncoes": (
            f"{df['cenario_id'].nunique()} cenarios; equipamentos por cenario: "
            f"{df.groupby('cenario_id')['equipamento'].count().to_dict()}"),
    }


relatorio = {**verificar(cargas, ["cenario_id", "equipamento"]), **dominio_cargas(cargas)}
pd.DataFrame(relatorio.items(), columns=["verificacao", "resultado"])


,verificacao,resultado
0,1. chaves/duplicidade,0 linhas duplicadas; 0 duplicadas na chave ('c...
1,2. ausencia/codigos,0 celulas nulas
2,3. tipos/unidades,"cenario_id=str, cenario=str, duracao_h=float64..."
3,4. dominio/plausibilidade,potencia min = 0.02 kW (todas > 0: True); qtd ...
4,5. cobertura/juncoes,3 cenarios; equipamentos por cenario: {'posto_...


In [4]:
problemas = {
    "equipamento repetido no cenario": int(cargas.duplicated(subset=["cenario_id", "equipamento"]).sum()),
    "valores nulos": int(cargas.isna().sum().sum()),
    "potencia ou quantidade invalida": int(((cargas["potencia_kW"] <= 0) | (cargas["qtd"] < 1)).sum()),
}

decisoes = pd.DataFrame([
    {"problema": "Equipamento repetido no cenario",
     "deteccao": "duplicated(subset=[cenario_id, equipamento])",
     "afetados": problemas["equipamento repetido no cenario"],
     "regra": "se existir, somar em uma unica linha",
     "consequencia": "composicao da carga preservada"},
    {"problema": "Valores nulos",
     "deteccao": "isna()", "afetados": problemas["valores nulos"],
     "regra": "descartar a linha (nunca imputar carga)",
     "consequencia": "nenhuma linha removida"},
    {"problema": "Potencia <= 0 ou quantidade < 1",
     "deteccao": "(potencia_kW <= 0) | (qtd < 1)",
     "afetados": problemas["potencia ou quantidade invalida"],
     "regra": "revisar a fonte antes de calcular",
     "consequencia": "nenhuma linha corrigida"},
])

cargas.to_csv(PROC / "cargas_criticas_por_cenario.csv", index=False)
print("Sem alteracoes" if sum(problemas.values()) == 0 else f"ATENCAO: {problemas}")
decisoes


Sem alteracoes


,problema,deteccao,afetados,regra,consequencia
0,Equipamento repetido no cenario,"duplicated(subset=[cenario_id, equipamento])",0,"se existir, somar em uma unica linha",composicao da carga preservada
1,Valores nulos,isna(),0,descartar a linha (nunca imputar carga),nenhuma linha removida
2,Potencia <= 0 ou quantidade < 1,(potencia_kW <= 0) | (qtd < 1),0,revisar a fonte antes de calcular,nenhuma linha corrigida


## 4. Indicador

```
Pcritica = soma(potencia_i * qtd_i)      [kW]
Edemanda = Pcritica * duracao            [kWh]
```


In [5]:
pcritica = (cargas.groupby(["cenario_id", "cenario"], as_index=False)["subtotal_kW"]
            .sum().rename(columns={"subtotal_kW": "Pcritica_kW"}))
duracao = cargas.groupby(["cenario_id", "cenario"], as_index=False)["duracao_h"].first()
pcritica = pcritica.merge(duracao, on=["cenario_id", "cenario"])
pcritica["Edemanda_kWh"] = (pcritica["Pcritica_kW"] * pcritica["duracao_h"]).round(3)
pcritica


,cenario_id,cenario,Pcritica_kW,duracao_h,Edemanda_kWh
0,posto_vacinas,Sala de vacinas (posto de saude),0.5,8.0,4.0
1,residencia_essencial,Residencia - circuito essencial,0.91,6.0,5.46
2,telecom_erb,Estacao radio-base (telecom),1.2,4.0,4.8


## 5. Visualizacao - V1

Barras **horizontais** (small multiples): um painel por cenario, uma barra por equipamento, `Pcritica` anotada em cada painel.


In [6]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

reais = [cen for cen in cfg["cenarios"] if cen["cargas"]]
fig = make_subplots(rows=1, cols=len(reais), subplot_titles=[cen["nome"] for cen in reais],
                    horizontal_spacing=0.09)

for i, cen in enumerate(reais, 1):
    itens = list(reversed(cen["cargas"]))
    labels = [x["equipamento"] for x in itens]
    vals = [x["potencia_kW"] * x["qtd"] for x in itens]
    pcrit = sum(vals)
    fig.add_trace(go.Bar(
        x=vals, y=labels, orientation="h", marker_color="#0B6E4F",
        text=[f"{v:.2f} kW" for v in vals], textposition="outside", showlegend=False,
        hovertemplate="%{y}<br>Contribuicao: %{x:.2f} kW<extra></extra>"), row=1, col=i)
    xref = "x domain" if i == 1 else f"x{i} domain"
    fig.add_annotation(text=f"Pcritica = {pcrit:.2f} kW", x=0.5, y=1.16, xref=xref,
                       yref="y domain", showarrow=False, font=dict(size=11, color="#0B6E4F"))

fig.update_xaxes(title_text="Potencia simultanea (kW)", rangemode="tozero")
fig.update_layout(title="V1. Quais cargas formam a demanda critica?",
                  template="plotly_white", height=560)
fig.show()
fig.write_html(VISTAS / "v1_carga_critica.html", include_plotlyjs="cdn")


## 6. Por que este grafico e a melhor opcao?

**Tarefa visual:** comparar **magnitudes nominais** entre poucas categorias de **rotulos longos** (nomes de equipamentos), dentro de cada cenario.

- **Barras horizontais** usam o **comprimento** como canal visual - o mais preciso para estimar magnitude (Cleveland e McGill). O tamanho e lido a partir de uma linha de base comum (zero), sem distorcao.
- **Rotulos longos** ("Ar-condicionado de emergencia") ficam legiveis na horizontal; na vertical teriam de ser girados.
- **Small multiples** (1 painel por cenario) evitam sobrepor cenarios diferentes e ainda permitem compara-los entre si.
- A **anotacao da `Pcritica`** liga o grafico ao indicador: composicao e total no mesmo lugar.

**Alternativas descartadas**

| Alternativa | Por que nao |
|---|---|
| Pizza / rosca | Comparacao angular imprecisa; pior com rotulos longos e valores proximos. |
| Barras verticais | Rotulos precisariam ser girados; leitura pior. |
| Barra empilhada (1 cenario = 1 barra) | Some as contribuicoes, mas **impede comparar equipamentos** entre si. |
| Treemap | Usa **area**, canal menos preciso que comprimento; dificulta a comparacao fina. |


## 7. Leitura, decisao e limitacoes

- **Leitura:** o `Ar-condicionado de emergencia` domina a carga de telecom (0,80 kW = 67% da `Pcritica` de 1,20 kW); na residencia, o maior item e a `Bomba d'agua` (0,50 kW).
- **Decisao apoiada:** priorizar/religar em cascata os equipamentos de maior subtotal quando a autonomia for curta.
- **Limite:** as cargas sao **hipotese declarada**, nao medicao - nao servem para dimensionar instalacao real. Cargas de partida (inrush) e fator de potencia nao estao incluidos.
